## Running Conditional VGAE

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation; $z \in \mathbb{R}^{N \times D}$: representation; interpretability as "trajectory / gradient")
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle
import gzip
import tifffile

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import scFates as scf

import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, configs, dataset, trajectory
from models import vgae, model_train
from torch_geometric.loader import DataLoader

In [ ]:
from importlib import reload
reload(IO)
reload(utils)
reload(plot)
reload(configs)
reload(dataset)
reload(vgae)
reload(model_train)

### VGAE (Xenium-DESI integration)

In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
metadata_path = '../data/sample_metadata.csv'

n_desi = 100
n_aux = 10
run_single_sample = True

if run_single_sample:
    sample_id = 'NIH_F5'
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=n_desi
    )
else:
    # All samples
    sample_ids = sorted([
        f for f in os.listdir(xenium_path) 
        if os.path.isdir(os.path.join(xenium_path, f))
    ])
    
    metadata_df = pd.read_csv(metadata_path, index_col=[0])
    metadata_cols = ['SEX', 'BMI']
    metadata_df = metadata_df[metadata_cols]
    metadata_df['SEX'] = metadata_df['SEX'].apply(lambda x: 0 if x == 'F' else 1)
    
    adatas = []
    for sample_id in sample_ids:
        adata = IO.load_multiomics(
            sample_id, xenium_path, desi_path,
            n_features=n_desi, mdata_df = metadata_df
        )
        adatas.append(adata)

gc.collect()


In [ ]:
adata.obsm['X_aux'].shape

#### Model training: 
- $z_i \mid u_i \sim \mathcal{N}(f_{\lambda_{\mu}}(u_i), f_{\lambda_{\sigma}}(u_i))$
- $x_i \mid z_i; \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i; \mathbf{A}), \theta_g)$

In [ ]:
if run_single_sample:
    loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=16)
    graph_data = loader.load_graphs([adata])
    train_dl = DataLoader(graph_data, shuffle=True)
else:
    # Randomly split samples for training / test (per gender)
    split_ratio = 0.6
    female_indices = np.arange(len(adatas)//2)
    male_indices = np.arange(len(adatas)//2, len(adatas))
    
    train_indices = list(np.random.choice(female_indices, size=int(len(female_indices)*split_ratio), replace=False)) + \
                    list(np.random.choice(male_indices, size=int(len(male_indices)*split_ratio), replace=False))

    loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=10)
    train_data = loader.load_graphs([adatas[i] for i in train_indices])
    train_dl = DataLoader(train_data, batch_size=16, shuffle=True)
    
    del female_indices, male_indices
gc.collect()

In [ ]:
torch.manual_seed(0)
lr = 1e-3
device = torch.device('cuda')

n_epochs = 200
n_hidden = 64
n_embedding = 16
n_latent = 8
n_aux = adata.obsm['X_aux'].shape[1]
n_covariate = adata.obsm['X_s'].shape[1] if 'X_s' in adata.obsm_keys() \
              else 0

train_configs = configs.set_train_configs(
    n_epochs=n_epochs, 
    lr=lr, 
    gamma=1., 
    annealing=False,
    device=device
)

model_configs = configs.set_model_configs(
    c_in=adata.shape[1], c_aux=n_aux, c_covariate=n_covariate, 
    c_hidden=n_hidden, c_latent=n_latent,
    beta=0.999, k_hop=3, dropout=0.5, act=nn.SiLU(), 

    # Debug cross-attention:
    # embed_option='attn',
    # c_embedding=n_embedding,
    
    device=device, 
) 

In [ ]:
pyro.clear_param_store()
model = vgae.VGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_vgae(model, train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
display_trajectory = False
k = 30
n_subgraphs = 4
device = torch.device('cpu')
dist_metric = 'euclidean'

if run_single_sample:
    preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
    
    # Trajectory inference
    adata.obsm['X_z'] = preds.qz
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        dist_metric=dist_metric,
        k=0
    )

    # Spatial visualization
    if display_trajectory:
        # Factor disentanglement
        z_corr = np.corrcoef(adata.obsm['X_z'].T)
        g = sns.clustermap(z_corr, cmap='RdBu_r')
            
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n LYNX'
        )
        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Pseudotime\n'+'LYNX'
        )
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='tab10', size=20, img=False,
            title='Zonation\n'+'LYNX'
        )

In [ ]:
sq.pl.spatial_scatter(
    adata, color='t',
    img=False, size=20, cmap='RdBu'
)

In [ ]:
# Plot individual `pz`
pz = preds.pz
pz_labels = ['z_'+str(i) for i in range(adata.obsm['X_z'].shape[1])]
for i in range(len(pz_labels)):
    adata.obs[pz_labels[i]] = pz[:, i]

sq.pl.spatial_scatter(
    adata,
    color=pz_labels,
    img=False, size=20, 
    cmap='Reds'
)

In [ ]:
# Plot individual `qz`
qz_labels = ['z_'+str(i) for i in range(adata.obsm['X_z'].shape[1])]
for i in range(len(qz_labels)):
    adata.obs[qz_labels[i]] = adata.obsm['X_z'][:, i]

sq.pl.spatial_scatter(
    adata,
    color=qz_labels,
    img=False, size=20, 
    cmap='Reds'
)

In [ ]:
torch.save(model.state_dict(), '../results/multi_new.pt'.format(n_latent))                                    
del adatas, train_data, train_dl
gc.collect()

#### Ablation study

Disentanglement w/ 
- Randomized `u` (seems no effects vs. correct `u`
- Pseudo-ground-truth, one-hot-encoded zonation assignments as `u`

In [ ]:
import scFates as scf

In [ ]:
# Perturb u with white noise
adata.obsm['X_aux'] = np.random.normal(scale=1e-2, size=(adata.shape[0], adata.obsm['X_aux'].shape[1]))
loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8)
graph_data = loader.load_graphs([adata])
train_dl = DataLoader(graph_data, shuffle=True)

In [ ]:
# Perturb u with "ground-truth" labels
adata.obsm['X_aux'] = F.one_hot(
    torch.tensor(adata.obs['milestones'].to_numpy())
).numpy()
loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8)
graph_data = loader.load_graphs([adata])
train_dl = DataLoader(graph_data, shuffle=True)

In [ ]:
lr = 1e-3
device = torch.device('cuda')

n_epochs = 500
n_hidden = 64
n_latent = 8
n_aux = adata.obsm['X_aux'].shape[1]
n_covariate = adata.obsm['X_s'].shape[1] if 'X_s' in adata.obsm_keys() else 0

train_configs = configs.set_train_configs(
    n_epochs=n_epochs, 
    lr=lr, 
    gamma=1.,
    annealing=False,
    device=device
)

model_configs = configs.set_model_configs( 
    c_in=adata.shape[1], c_aux=n_aux, c_covariate=n_covariate, 
    c_hidden=n_hidden, c_latent=n_latent,
    beta=1, k_hop=3, dropout=0.5, act=nn.SiLU(),
    device=device, 
) 

In [ ]:
pyro.clear_param_store()
model = vgae.VGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_vgae(model, train_dl, train_configs)

In [ ]:
k = 30
n_subgraphs = 8
device = torch.device('cpu')
dist_metric = 'euclidean'

preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
adata.obsm['X_z'] = preds.qz

# Check factor disentanglement
z_corr = np.corrcoef(adata.obsm['X_z'].T)
g = sns.clustermap(z_corr, cmap='RdBu_r')

# Trajectory inference
trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    dist_metric=dist_metric,
    k=0
)

# Spatial visualization
plot.disp_trajectory(
    adata, 
    title='Spatial Gradients\n LYNX w/ corrupted U'
)

sq.pl.spatial_scatter(
    adata, color='t', 
    cmap='RdBu', size=20, img=False,
    title='Pseudotime\n'+'LYNX w/ corrupted U'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n'+'LYNX w/ corrupted U'
)

In [ ]:
# Plot individual `pz`
pz = model.pz_u(torch.tensor(adata.obsm['X_aux']).float()).detach().cpu().numpy()
qz_labels = ['z_'+str(i) for i in range(adata.obsm['X_z'].shape[1])]
for i in range(len(qz_labels)):
    adata.obs[qz_labels[i]] = pz[:, i]

sq.pl.spatial_scatter(
    adata,
    color=qz_labels,
    img=False, size=20, 
    cmap='Reds'
)

In [ ]:
# Plot individual `qz`
qz_labels = ['z_'+str(i) for i in range(adata.obsm['X_z'].shape[1])]
for i in range(len(qz_labels)):
    adata.obs[qz_labels[i]] = adata.obsm['X_z'][:, i]

sq.pl.spatial_scatter(
    adata,
    color=qz_labels,
    img=False, size=20, 
    cmap='Reds'
)

#### Held-out validation - metrics

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scFates as scf
import igraph

import json
import squidpy as sq

import pyro
from pyro.infer import SVI, Trace_ELBO, infer_discrete

from util import IO, utils, gen_graph, trajectory
from models import vgae, model_train, dataset
from torch_geometric import utils as pyg_utils
from torch_geometric.data import DataLoader

from importlib import reload
reload(utils)
reload(gen_graph)
reload(dataset)
reload(trajectory)

In [ ]:
def evaluate_elbo(model, dataloader, device=torch.device('cpu')):
    from pyro.optim import Adam
    optimizer = Adam({"lr": 1.0e-3})  # dummy optimizer
    
    model = model.to(device)
    elbo = Trace_ELBO()
    svi = SVI(model.model, model.guide, optimizer, elbo)
    elbos = []
    
    for data in dataloader:
        x = data.x.to(device).float()
        u = data.u.to(device).float()
        edge_index = data.edge_index.to(device)
        elbo = svi.evaluate_loss(x, u, edge_index)

        elbos.append(elbo / x.shape[0])

    return elbos

def evaluate_kl(model, dataloader, device=torch.device('cpu')):
    from pyro.optim import Adam
    from pyro.infer import TraceMeanField_ELBO
    from pyro import poutine
    
    optimizer = Adam({"lr": 1.0e-3})  # dummy optimizer
    
    model = model.to(device)
    elbo = TraceMeanField_ELBO()
    svi = SVI(model.model, model.guide, optimizer, elbo)
    
    kl_divs = []
    for data in dataloader:
        x = data.x.to(device).float()
        u = data.u.to(device).float()
        edge_index = data.edge_index.to(device)

        with poutine.scale(scale=1e-7):
            kl_div = svi.evaluate_loss(x, u, edge_index)
            kl_divs.append(kl_div)

    return kl_divs

def cell_type_dynamics(
    adata,
    cell_type, 
    show_fig=False
):
    """
    Dynamics of cell-type proportion along discrete zonations
    """
    assert 'milestones' in adata.obs.columns and 'cell_type' in adata.obs.columns
    assert cell_type in adata.obs['cell_type'].unique()

    zonation_states = np.unique(adata.obs['milestones'])
    dynamics = []
    for state in zonation_states:
        summary = adata[adata.obs['milestones'] == state].obs['cell_type'].value_counts()
        if cell_type in summary.index:
            target_count = summary[cell_type]
            total_count = summary.sum()
            dynamics.append(target_count / total_count)
        else:
            dynamics.append(0)

    if show_fig:
        fig, ax = plt.subplots(figsize=(4, 1))
        ax.plot(zonation_states, dynamics, '.--', c='blue')
        ax.set_title(cell_type)
        plt.show()
    
    return dynamics

In [ ]:
# Load validation data
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
n_aux = 10
n_latent = 8
n_hidden = 16

sample_ids = np.array(sorted([
    f for f in os.listdir(xenium_path) 
    if os.path.isdir(os.path.join(xenium_path, f))
]))

train_ids = np.array(sample_ids[train_indices])
val_ids = np.setdiff1d(np.arange(len(sample_ids)), train_indices)

adatas_train = []
for sample_id in train_ids:
    print('Loading training sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_train.append(adata)
    del adata_desi
    gc.collect()
    
adatas_val = []
for sample_id in val_ids:
    print('Loading val sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_val.append(adata)
    del adata_desi
    gc.collect()

del sample_id

In [ ]:
# Load models
device = torch.device('cuda')
model_configs = configs.set_model_configs(
    device=device, act=nn.SiLU(), beta=0.5, 
    c_in=adata.shape[1], c_aux=n_aux, c_hidden=8, c_latent=n_latent,
    k_hop=3, dropout=0.5
) 

model_normal = vgae.VGAE(configs=model_configs, device=device)
model_normal.load_state_dict(torch.load('../results/multi_{}.pt'.format(model_configs.c_latent)))

Fitting metrics:
- Neg. ELBO
- Reconstruction NLL
- KL div.

In [ ]:
# Create subgraph-batched graph data
loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8)
train_data = loader.load_graphs(adatas_train)
val_data = loader.load_graphs(adatas_val)
train_dl = DataLoader(train_data)
val_dl = DataLoader(val_data)

del train_data, val_data
gc.collect()

In [ ]:
train_elbos_normal = evaluate_elbo(model_normal, train_dl, device=device)
val_elbos_normal = evaluate_elbo(model_normal, val_dl, device=device)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# elbo_df = pd.DataFrame({
#     'ELBO': train_elbos_normal + train_elbos_vMF + \
#             val_elbos_normal + val_elbos_vMF,
    
#     'Label': ['train']*len(train_elbos_normal)*2 + \
#              ['val']*len(val_elbos_normal)*2,
    
#     'Prior': ['Normal']*len(train_elbos_normal) + \
#              ['vMF']*len(train_elbos_vMF) + \
#              ['Normal']*len(val_elbos_normal) + \
#              ['vMF']*len(val_elbos_vMF)
# })

# fig, ax = plt.subplots(figsize=(6, 4))
# ax = sns.barplot(x='Label', y='ELBO', data=elbo_df, hue='Prior')
# ax.spines[['right', 'top']].set_visible(False)
# sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
# ax.set_title('ELBO')
# plt.show()

In [ ]:
train_kls_normal = evaluate_kl(model_normal, train_dl, device=device)
val_kls_normal = evaluate_kl(model_normal, val_dl, device=device)
gc.collect()
torch.cuda.empty_cache()

#### Held-out validation - antibody

In [ ]:
import scFates as scf
import igraph
import json
import squidpy as sq

import warnings
warnings.filterwarnings('ignore')

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
metadata_path = '../data/sample_metadata.csv'

sample_ids = np.array(sorted([
    f for f in os.listdir(xenium_path) 
    if os.path.isdir(os.path.join(xenium_path, f))
]))

metadata_df = pd.read_csv(metadata_path, index_col=[0])
metadata_cols = ['SEX', 'BMI']
metadata_df = metadata_df[metadata_cols]
metadata_df['SEX'] = metadata_df['SEX'].apply(lambda x: 0 if x == 'F' else 1)

train_ids = sample_ids[train_indices]
val_ids = sample_ids[np.setdiff1d(np.arange(len(sample_ids)), train_indices)]


In [ ]:
# Load model
n_latent = 8
n_covariate = metadata_df.shape[-1]
n_aux = 10
n_hidden = 64

adata = IO.load_xenium(os.path.join(xenium_path, sample_ids[0]))
n_features = adata.shape[1]
device = torch.device('cpu')
del adata

model_configs = configs.set_model_configs(
    c_in=n_features, c_aux=n_aux, c_covariate=n_covariate, 
    c_hidden=n_hidden, c_latent=n_latent,
    beta=1, k_hop=3, dropout=0.5, act=nn.SiLU(), 
    device=device, 
) 

pyro.clear_param_store()
model = vgae.VGAE(model_configs, device=train_configs.device)
model.load_state_dict(torch.load('../results/normal_multi_new.pt'))

# model.load_state_dict(torch.load('../results/normal_multi_new.pt'))


In [ ]:
display = True
k = 30
n_subgraphs = 8
device = torch.device('cpu')
dist_metric = 'euclidean'


for i, sample_id in enumerate(sample_ids):
# for i, sample_id in enumerate(val_ids):
    # ----------------------------------------
    print('Visual checks on trajectory of {}...'.format(sample_id))
    print('Computing latent representation...')

    # Load adata
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path,
        n_pcs=n_aux, mdata_df = metadata_df,
        verbose=True
    )

    preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
    px = preds.px
    qz = preds.pz
    adata.obsm['X_z'] = qz
    
    scf.tl.curve(
        adata, 
        use_rep='X_z',
        Nodes=n_latent, 
        epg_extend_leaves=True,
        ndims_rep=n_latent
    )

    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        dist_metric='euclidean', 
        k=0
    )

    if display:
        # Reconstruction quality
        indices = np.random.choice(
            adata.shape[1], size=300, replace=False
        )
        plt.figure(figsize=(6, 4))
        plt.scatter(
            adata.X.A[:, indices].flatten(),
            px[:, indices].flatten(), 
            s=.5
        )
        plt.xlabel('X')
        plt.ylabel('X_reconst')
        plt.show()

        # Latent representation
        # sc.pp.neighbors(adata, use_rep='X_z')
        # sc.tl.umap(adata)
    
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n ({})'.format(sample_id)
        )
        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Spatial Gradients\n ({})'.format(sample_id)
        )
        
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='tab20', size=20, img=False,
            title='Zonation\n ({})'.format(sample_id)
        )
    
    # ----------------------------------------
    print('\t Loading paired Ab images...')
    filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
    img = tifffile.imread(filename)[1:]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
    ax1.imshow(img[1], cmap='magma')
    ax1.set_title('Peri-central')
    ax1.axis('off')

    ax2.imshow(img[2], vmax=0.8, cmap='magma')
    ax2.set_title('Peri-portal')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

    # Save latent
    np.save('../results/lynx_{0}_{1}.npy'.format(sample_id, n_latent), qz)

    print('====================================\n\n')
    del adata, img
    gc.collect()


In [ ]:
# Latent representation
sc.pp.neighbors(adata, use_rep='X_z')
sc.tl.umap(adata)

scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent)))
sc.pl.umap(
    adata, color='t',
    vmin=np.quantile(adata.obs['t'], .1),
    vmax=np.quantile(adata.obs['t'], .9),
    cmap='RdBu'
)

sq.pl.spatial_scatter(
    adata, color='t', 
    cmap='RdBu', size=20, img=False,
    vmin=np.quantile(adata.obs['t'], .1),
    vmax=np.quantile(adata.obs['t'], .9),
    title='Pseudotime'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab20', size=20, img=False,
    title='Zonation'
)

In [ ]:
filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
img = tifffile.imread(filename)[1:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(img[1], cmap='magma')
ax1.set_title('Peri-central')
ax1.axis('off')

ax2.imshow(img[2], cmap='magma')
ax2.set_title('Peri-portal')
ax2.axis('off')

plt.tight_layout()
plt.show()


---